In [2]:
import os
import sys
import fastf1
from datetime import datetime
import pandas as pd
import numpy as np
import streamlit as st

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error


In [3]:
sys.path.append(os.path.abspath(".."))
from fonctions_predictions import initialize_feature_df, encoding_label, calculate_podium_proba

In [4]:
actual_date = pd.to_datetime("2026-03-14, 12:00:00")

In [5]:
def get_calendar(year):
    calendar = fastf1.get_event_schedule(year)
    
    df_calendar = calendar.copy()

    colonnes = [
        'RoundNumber','Country','Location','EventDate','EventName',
        'EventFormat','Session1','Session1DateUtc','Session2',
        'Session2DateUtc','Session3','Session3DateUtc','Session4',
        'Session4DateUtc','Session5','Session5DateUtc'
    ]
    
    df_calendar = df_calendar[colonnes]

    sessions_cols = ['Session1DateUtc', 'Session2DateUtc', 'Session3DateUtc','Session4DateUtc', 'Session5DateUtc']
    
    df_calendar['EventDate'] = pd.to_datetime(df_calendar['EventDate'])
    
    for col in sessions_cols:
        df_calendar[col] = (pd.to_datetime(df_calendar[col], utc=True).dt.tz_convert("Europe/Paris")).dt.tz_localize(None)
    return df_calendar

In [6]:
df_calendar = get_calendar(2026)

req         WARNING 	DEFAULT CACHE ENABLED! (276.83 MB) C:\Users\mike5\AppData\Local\Temp\fastf1


In [7]:
futur_events = df_calendar[df_calendar['Session5DateUtc'] > actual_date]
next_index = futur_events['Session5DateUtc'].idxmin()
next_event = df_calendar.loc[next_index]

past_events = df_calendar[df_calendar['Session5DateUtc'] < actual_date]
last_index = past_events['Session5DateUtc'].idxmax()
last_event = df_calendar.loc[last_index]

In [8]:
df_master = initialize_feature_df(2026, 1)

core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	No lap data for driver 81
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 81)
req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 22 drivers: ['63', '12', '16', '44', '1', '3', '87', '41', '5', '10', '31', '23', '30', '43', '55', '11', '18', '14', '77', '6', '81', '27']
core           INFO 	Loading data for Australian Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req

Récupération météo réelle via FastF1...


In [9]:
df_next_gp = initialize_feature_df(2026, next_event['RoundNumber'])

core           INFO 	Loading data for Chinese Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for weather_data
core        WARNING 	Driver 12 completed the race distance 00:00.022000 before the recorded end of the session.
core           INFO 	Finished loading data for 22 drivers: ['12', '63', '44', '16', '87', '10', '30', '6', '55', '43', '27', '41', '77', '31', '11', '3', '14', '18', '81', '1', '5', '23']
core           INFO 	Loading data for Chinese Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached

Récupération météo réelle via FastF1...


req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['63', '12', '16', '44', '1', '3', '87', '41', '5', '10', '31', '23', '30', '43', '55', '11', '18', '14', '77', '6', '81', '27']


In [11]:
df_next_gp

,Abbreviation,TeamName,qualif_pos,race_finish_pos,last_qualif_pos,EventName,RoundNumber,Year,is_race_incident,qualif_time,GapFromPole_pct,is_incident_quali,topspeed_kmh_qualif,constructor_pos,team_performance_lastgp,air_temp_forecast,track_temp_forecast,rain_proba_forecast
0,ANT,Mercedes,1.0,1.0,2.0,Chinese Grand Prix,2,2026,0,92.064,0.000,0,334.0,1,43.0,15.57,22.62,0
1,RUS,Mercedes,2.0,2.0,1.0,Chinese Grand Prix,2,2026,0,92.286,0.241,0,341.0,1,43.0,15.57,22.62,0
2,HAM,Ferrari,3.0,3.0,7.0,Chinese Grand Prix,2,2026,0,92.415,0.381,0,335.0,2,27.0,15.57,22.62,0
3,LEC,Ferrari,4.0,4.0,4.0,Chinese Grand Prix,2,2026,0,92.428,0.395,0,337.0,2,27.0,15.57,22.62,0
4,BEA,Haas F1 Team,10.0,5.0,12.0,Chinese Grand Prix,2,2026,0,93.197,1.231,0,334.0,3,6.0,15.57,22.62,0
5,GAS,Alpine,7.0,6.0,14.0,Chinese Grand Prix,2,2026,0,92.873,0.879,0,340.0,4,1.0,15.57,22.62,0
6,LAW,Racing Bulls,14.0,7.0,8.0,Chinese Grand Prix,2,2026,0,93.765,1.848,0,337.0,5,4.0,15.57,22.62,0
7,HAD,Red Bull Racing,9.0,8.0,3.0,Chinese Grand Prix,2,2026,0,93.121,1.148,0,338.0,6,8.0,15.57,22.62,0
8,SAI,Williams,17.0,9.0,21.0,Chinese Grand Prix,2,2026,0,94.317,2.447,0,338.0,7,0.0,15.57,22.62,0
9,COL,Alpine,12.0,10.0,16.0,Chinese Grand Prix,2,2026,0,93.357,1.404,0,343.0,4,1.0,15.57,22.62,0


In [10]:
# --- 2. Récupérer le momentum (last_qualif_pos) du GP précédent ---
mapping_last_quali = df_master[df_master['RoundNumber'] == next_event['RoundNumber']-1].set_index('Abbreviation')['qualif_pos'].to_dict()
df_next_gp['last_qualif_pos'] = df_next_gp['Abbreviation'].map(mapping_last_quali)

In [12]:
df_master, df_next_gp, encoders = encoding_label(df_master, df_next_gp)

# Données retenues pour l'entrainement du modèle
features = [
    'Abbreviation', 'TeamName', 'qualif_pos', 'last_qualif_pos', 
    'GapFromPole_pct', 'is_incident_quali', 'topspeed_kmh_qualif', 
    'constructor_pos', 'team_performance_lastgp', 
    'air_temp_forecast', 'track_temp_forecast', 'rain_proba_forecast'
]

In [13]:
 # On entraîne sur l'historique (en filtrant les abandons techniques pour plus de pureté)
train_set = df_master[df_master['is_race_incident'] == 0]

X_train = train_set[features].copy()
y_train = train_set['race_finish_pos'].copy()

X_predict = df_next_gp[features].copy()

In [ ]:
model = GradientBoostingRegressor(
                n_estimators=250, 
                learning_rate=0.05, 
                max_depth=3, 
                random_state=42
            )

# --- 5. Entraînement ---
model.fit(X_train, y_train)

predictions = model.predict(X_predict)

In [15]:
df_next_gp['predicted_pos'] = predictions
results = df_next_gp[['Abbreviation', 'predicted_pos']].sort_values(by='predicted_pos')

# Calcul de la probabilité de podium
results['Podium_Proba_pct'] = results['predicted_pos'].apply(calculate_podium_proba)

# --- 7. Traduction ---
# On utilise l'encodeur stocké précédemment pour retrouver le nom du pilote
results['Driver'] = encoders['Abbreviation'].inverse_transform(results['Abbreviation'])
results['RoundNumber'] = df_next_gp['RoundNumber'].iloc[0]
results = results.sort_values(by='predicted_pos')

# --- 8. Nettoyage final ---
results = results.reset_index(drop=True)
results.index = results.index + 1
results.index.name = 'Pos'

In [16]:
results

,Abbreviation,predicted_pos,Podium_Proba_pct,Driver,RoundNumber
Pos,,,,,
1,2,1.376983,98.6,ANT,2
2,18,1.538308,98.1,RUS,2
3,12,2.675167,83.9,LEC,2
4,9,2.918442,76.2,HAM,2
5,7,3.850412,33.2,GAS,2
6,3,4.083379,23.7,BEA,2
7,15,4.283457,17.3,OCO,2
8,6,4.341116,15.7,COL,2
9,10,8.046684,0.0,HUL,2


In [23]:
df_actual = df_next_gp[df_next_gp['RoundNumber'] == 2][['Abbreviation', 'race_finish_pos']]
df_actual

,Abbreviation,race_finish_pos
0,2,1.0
1,18,2.0
2,9,3.0
3,12,4.0
4,3,5.0
5,7,6.0
6,11,7.0
7,8,8.0
8,19,9.0
9,6,10.0


In [24]:
df_compare = pd.merge(
    results,
    df_actual,
    on="Abbreviation"
)
df_compare = df_compare.reset_index()
df_compare['index'] = df_compare['index']+1
df_compare

,index,Abbreviation,predicted_pos,Podium_Proba_pct,Driver,RoundNumber,race_finish_pos
0,1,2,1.376983,98.6,ANT,2,1.0
1,2,18,1.538308,98.1,RUS,2,2.0
2,3,12,2.675167,83.9,LEC,2,4.0
3,4,9,2.918442,76.2,HAM,2,3.0
4,5,7,3.850412,33.2,GAS,2,6.0
5,6,3,4.083379,23.7,BEA,2,5.0
6,7,15,4.283457,17.3,OCO,2,14.0
7,8,6,4.341116,15.7,COL,2,10.0
8,9,10,8.046684,0.0,HUL,2,11.0
9,10,4,9.099962,0.0,BOR,2,21.0


In [25]:
mae_rank = mean_absolute_error(df_compare['race_finish_pos'], df_compare['index'])
print(f"MAE RANK : {mae_rank}")

# MAE sur la valeur brute -> Précision du "Cerveau"
mae_raw = mean_absolute_error(df_compare['race_finish_pos'], df_compare['predicted_pos'])
print(f"MAE RAW : {mae_raw}")


MAE RANK : 3.6363636363636362
MAE RAW : 3.9618712650017387
